<a href="https://colab.research.google.com/github/MBR4V0/Python/blob/main/PySpark.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Agregação Avançada usando Spark SQL**

In [ ]:
import sys
from awsglue.utils import getResolvedOptions
from pyspark.context import SparkContext
from awsglue.context import GlueContext
from awsglue.job import Job

args = getResolvedOptions(sys.argv, ['JOB_NAME'])
sc = SparkContext()
glueContext = GlueContext(sc)
spark = glueContext.spark_session
job = Job(glueContext)
job.init(args['JOB_NAME'], args)

# 1. Lê os dados de origem
df_vendas = glueContext.create_dynamic_frame.from_catalog(
    database="e_commerce",
    table_name="vendas"
).toDF()

# 2. Cria uma View Temporária para usar SQL
df_vendas.createOrReplaceTempView("vendas_temp")

# 3. Executa uma Query SQL de Agregação (Ex: Faturamento total por categoria de produto)
df_resultado_sql = spark.sql("""
    SELECT
        categoria_produto,
        COUNT(id_pedido) as total_pedidos,
        SUM(valor_item) as faturamento_total,
        CURRENT_DATE() as data_processamento
    FROM vendas_temp
    WHERE status_pagamento = 'APROVADO'
    GROUP BY categoria_produto
    ORDER BY faturamento_total DESC
""")

# 4. Grava o relatório gerado no S3
from awsglue.dynamicframe import DynamicFrame
df_final_glue = DynamicFrame.fromDF(df_resultado_sql, glueContext, "df_final_glue")

glueContext.write_dynamic_frame.from_options(
    frame = df_final_glue,
    connection_type = "s3",
    connection_options = {"path": "s3://meu-bucket/relatorios/faturamento_categoria/"},
    format = "parquet"
)

job.commit()